In [ ]:
"""
Week 1: IoT Telemetry Ingestion and Signal Processing

Author      : Subhashree Behera
Project     : Predictive Maintenance (Infotact Internship)
Dataset     : AI4I 2020 Predictive Maintenance Dataset (fused with weather data)

═══════════════════════════════════════════════════════════════

Objective:
    Engineer rolling-window statistical features from industrial
    IoT sensor data to capture short-term machine behavior trends
    that indicate degradation or impending failure.

═══════════════════════════════════════════════════════════════

Steps Performed:
    Step 1  → Import required libraries (pandas, numpy)
    Step 2  → Load fused dataset (ai4i_fused.csv) — 10,000 rows × 21 columns
    Step 3  → Clean column names (strip whitespace)
    Step 4  → Verify zero null values
    Step 5  → Sort chronologically by date + UDI (mandatory for time-series)
    Step 6  → Define 5 sensor columns for feature engineering
    Step 7  → Set rolling window size = 5
    Step 8  → Compute rolling MEAN for all sensors (5 new features)
    Step 9  → Compute rolling STD  for all sensors (5 new features)
    Step 10 → Compute rolling VAR  for all sensors (5 new features)
    Step 11 → Verify output: 15 new features, shape 21 → 36 columns
    Step 12 → Save engineered dataset (ai4i_rolling_features.csv)

═══════════════════════════════════════════════════════════════

Sensors Processed:
    • Air temperature [K]
    • Process temperature [K]
    • Rotational speed [rpm]
    • Torque [Nm]
    • Tool wear [min]

Feature Engineering Output:
    • 5 sensors × 3 statistics (mean, std, var) = 15 new features
    • Original columns : 21
    • Final columns    : 36
    • Rows retained    : 10,000 (min_periods=1, no rows dropped)

Key Finding:
    Torque rolling std = 9.26 (normal) vs 11.85 (failure)
    → Machines show higher torque instability before failure.
    → This validates rolling features as meaningful predictors.

═══════════════════════════════════════════════════════════════

Purpose:
    Rolling statistics capture short-term sensor trends that
    single-point readings miss — essential for predicting failures
    before they occur rather than reacting after breakdown.

    These 15 features feed directly into the LightGBM classifier
    in Week 3 for imbalanced failure classification.
"""

In [ ]:
# Step 1: Import required libraries
# pandas -> data manipulation, numpy -> numerical operations
import pandas as pd
import numpy as np

In [ ]:
# Step 2: Load the fused dataset (sensors + weather, merged by date)
df = pd.read_csv('../data/ai4i_fused.csv')

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

In [ ]:
# Step 3: Strip any accidental leading/trailing spaces from column names
# This prevents bugs later when referencing columns by exact name
df.columns = df.columns.str.strip()

print("Columns cleaned. Sample:", list(df.columns[:5]))

In [ ]:
# Step 4: Check for null values before doing any feature engineering
# Rolling window calculations break if there are NaNs in the middle of data
print("Total null values:", df.isnull().sum().sum())

In [ ]:
# Step 5: Sort by date (and UDI as tiebreaker) to ensure correct time-order
# Rolling window ONLY makes sense if data is in chronological sequence (last 5 readings)
df = df.sort_values(by=['date', 'UDI']).reset_index(drop=True)

print("First 3 dates after sorting:", df['date'].head(3).tolist())

In [ ]:
# Step 6: Define which sensor columns we'll engineer rolling features for
# These are the 5 raw machine sensor readings from the AI4I dataset
sensors = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

print(f"Will engineer rolling features for {len(sensors)} sensors:")
for s in sensors:
    print(" -", s)

In [ ]:
# Step 7: Set the rolling window size
# window=5 means: for each row, look at the current + previous 4 readings
# min_periods=1 means: even the first few rows get a value (using fewer
# points), instead of NaN — avoids losing data at the start
window = 5

print(f"Rolling window size set to: {window}")

In [ ]:
# Step 8: Calculate rolling MEAN for each sensor
# This smooths out short-term noise and shows the recent average trend
for col in sensors:
    df[f'{col}_rolling_mean'] = df[col].rolling(window=window, min_periods=1).mean()

print("Shape after adding rolling means:", df.shape)
df[['Torque [Nm]', 'Torque [Nm]_rolling_mean']].head(7)

In [ ]:
# Step 9: Calculate rolling STD for each sensor
# This measures how much the readings fluctuate in the recent window higher
# std = more unstable/erratic sensor behavior
for col in sensors:
    df[f'{col}_rolling_std'] = df[col].rolling(window=window, min_periods=1).std().fillna(0)

print("Shape after adding rolling std:", df.shape)

In [ ]:
# Step 10: Calculate rolling VARIANCE for each sensor
# Variance = std squared. It's another way to measure spread,
# often used directly in ML models since it amplifies larger fluctuations
for col in sensors:
    df[f'{col}_rolling_var'] = df[col].rolling(window=window, min_periods=1).var().fillna(0)

print("Shape after adding rolling variance:", df.shape)

In [ ]:
# Step 11: Verify the final feature set
# We should have 5 sensors x 3 stats (mean, std, var) = 15 new columns
new_cols = [c for c in df.columns if 'rolling' in c]

print("Total new rolling features created:", len(new_cols))
print("Original columns: 21  ->  Final columns:", df.shape[1])
print("\nNew feature columns:")
for c in new_cols:
    print(" -", c)

In [ ]:
# Step 12: Save the feature-engineered dataset for Week 2/3 use
df.to_csv('../data/ai4i_rolling_features.csv', index=False)

print("Saved successfully!")
print("Final dataset shape:", df.shape)
print("Saved to: ../data/ai4i_rolling_features.csv")